# Phase 2: ML Attrition Model — Step 2.3: Model Comparison & Tuning

This notebook evaluates classification algorithms under multiple scenarios to identify the most robust predictive model for employee attrition. We analyze:
1. **Unweighted / Default Baseline**
2. **Balanced Class Weights** (Logistic Regression, Random Forest, XGBoost)
3. **Regularized & Balanced Models** (Random Forest and XGBoost with regularization to prevent overfitting)
4. **Threshold Tuning** on Logistic Regression (Balanced) to optimize the operational trade-off between Precision and Recall.

We track **Train Recall** alongside **Test Recall** to explicitly diagnose and confirm overfitting.

In [1]:
import os
import joblib
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

proc_dir = os.path.join("data", "processed")
print(f"Processed directory: {os.path.abspath(proc_dir)}")

## 1. Load Employees Data & Prepare Features

In [2]:
df_emp = pd.read_csv(os.path.join(proc_dir, "employees.csv"))

# Feature Engineering
df_emp["income_per_year_at_company"] = (df_emp["MonthlyIncome"] * 12) / (df_emp["YearsAtCompany"] + 1.0)
df_emp["years_since_last_promotion_gap"] = df_emp["YearsAtCompany"] - df_emp["YearsSinceLastPromotion"]
df_emp["overall_satisfaction_composite"] = (
    df_emp["EnvironmentSatisfaction"] 
    + df_emp["JobSatisfaction"] 
    + df_emp["RelationshipSatisfaction"] 
    + df_emp["WorkLifeBalance"]
) / 4.0
df_emp["experience_ratio"] = df_emp["YearsAtCompany"] / (df_emp["TotalWorkingYears"] + 1.0)

# Drop constants and leakage
constant_cols = ["EmployeeCount", "Over18", "StandardHours"]
id_cols = ["EmployeeNumber"]
target_col = "Attrition"
y = df_emp[target_col].map({"Yes": 1, "No": 0})
X = df_emp.drop(columns=constant_cols + id_cols + [target_col])

# Encoding
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=True)
bool_cols = X_encoded.select_dtypes(include=['bool']).columns
X_encoded[bool_cols] = X_encoded[bool_cols].astype(int)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.20, stratify=y, random_state=42
)
print(f"Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}")

## 2. Model Training & Evaluation Function

In [3]:
comparison_results = []

def evaluate_model(name, model_pipe, description):
    model_pipe.fit(X_train, y_train)
    y_train_pred = model_pipe.predict(X_train)
    y_test_pred = model_pipe.predict(X_test)
    y_test_prob = model_pipe.predict_proba(X_test)[:, 1]
    
    train_rec = recall_score(y_train, y_train_pred)
    test_rec = recall_score(y_test, y_test_pred)
    prec = precision_score(y_test, y_test_pred)
    f1 = f1_score(y_test, y_test_pred)
    auc = roc_auc_score(y_test, y_test_prob)
    
    comparison_results.append({
        "Model": name,
        "Scenario": description,
        "Train Recall": train_rec,
        "Test Recall": test_rec,
        "Precision": prec,
        "F1-Score": f1,
        "ROC-AUC": auc
    })

## 3. Run Scenario 1: Unweighted Models

In [4]:
evaluate_model("Logistic Regression", Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(random_state=42, max_iter=1000))
]), "Unweighted Baseline")

evaluate_model("Random Forest", Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", RandomForestClassifier(random_state=42))
]), "Unweighted Baseline")

evaluate_model("XGBoost", Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", XGBClassifier(random_state=42, eval_metric='logloss'))
]), "Unweighted Baseline")

## 4. Run Scenario 2: Balanced Models

In [5]:
neg_count = sum(y_train == 0)
pos_count = sum(y_train == 1)
scale_pos_w = neg_count / pos_count

evaluate_model("Logistic Regression (Balanced)", Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced'))
]), "Balanced Class Weights")

evaluate_model("Random Forest (Balanced)", Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", RandomForestClassifier(random_state=42, class_weight='balanced'))
]), "Balanced Class Weights")

evaluate_model("XGBoost (Balanced)", Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", XGBClassifier(random_state=42, eval_metric='logloss', scale_pos_weight=scale_pos_w))
]), "Balanced Class Weights")

## 5. Run Scenario 3: Regularized & Balanced Models
We introduce regularization parameters to restrict model complexity and verify whether overfitting is resolved.

In [6]:
evaluate_model("Random Forest (Regularized)", Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", RandomForestClassifier(random_state=42, class_weight='balanced', max_depth=4, min_samples_leaf=10))
]), "Regularized & Balanced")

evaluate_model("XGBoost (Regularized)", Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", XGBClassifier(random_state=42, eval_metric='logloss', scale_pos_weight=scale_pos_w,
                                 max_depth=3, min_child_weight=5, subsample=0.8, colsample_bytree=0.8))
]), "Regularized & Balanced")

## 6. Run Scenario 4: Logistic Regression Threshold Sweep
We sweep the classification threshold of the balanced Logistic Regression model from 0.30 to 0.50 (in steps of 0.05) to find the optimal trade-off.

In [7]:
# Retrieve trained balanced LR pipeline
lr_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced'))
])
lr_pipeline.fit(X_train, y_train)

lr_probs_train = lr_pipeline.predict_proba(X_train)[:, 1]
lr_probs_test = lr_pipeline.predict_proba(X_test)[:, 1]

thresholds = [0.30, 0.35, 0.40, 0.45]
for t in thresholds:
    train_pred = (lr_probs_train >= t).astype(int)
    test_pred = (lr_probs_test >= t).astype(int)
    
    train_rec = recall_score(y_train, train_pred)
    test_rec = recall_score(y_test, test_pred)
    prec = precision_score(y_test, test_pred)
    f1 = f1_score(y_test, test_pred)
    # ROC-AUC remains constant across threshold modifications
    auc = roc_auc_score(y_test, lr_probs_test)
    
    comparison_results.append({
        "Model": f"Logistic Regression (t={t:.2f})",
        "Scenario": "Threshold Sweep",
        "Train Recall": train_rec,
        "Test Recall": test_rec,
        "Precision": prec,
        "F1-Score": f1,
        "ROC-AUC": auc
    })

## 7. Final Model Comparison Registry

In [8]:
df_compare = pd.DataFrame(comparison_results)
print("=== Final Performance Registry ===")
print(df_compare.to_string(index=False))

=== Final Performance Registry ===
                         Model               Scenario  Train Recall  Test Recall  Precision  F1-Score  ROC-AUC
           Logistic Regression    Unweighted Baseline      0.500000     0.361702   0.653846  0.465753 0.810233
                 Random Forest    Unweighted Baseline      1.000000     0.106383   0.500000  0.175439 0.782539
                       XGBoost    Unweighted Baseline      1.000000     0.234043   0.611111  0.338462 0.770264
Logistic Regression (Balanced) Balanced Class Weights      0.794737     0.617021   0.341176  0.439394 0.800672
      Random Forest (Balanced) Balanced Class Weights      1.000000     0.255319   0.413793  0.315789 0.765828
            XGBoost (Balanced) Balanced Class Weights      1.000000     0.319149   0.483871  0.384615 0.759583
   Random Forest (Regularized) Regularized & Balanced      0.736842     0.680851   0.450704  0.542373 0.774830
         XGBoost (Regularized) Regularized & Balanced      1.000000     0.425

## 8. Rationale and Recommended Model

### Overfitting Analysis:
1. **Random Forest (Regularized)** successfully resolves the extreme overfitting. Train Recall drops to **73.68%** and Test Recall is **68.09%**. It achieves the highest F1-Score of **54.24%** and Precision of **45.07%**, making it a very strong contender.
2. **XGBoost (Regularized)** still overfits completely, reaching **100.0% Train Recall** and only **42.55% Test Recall**.

### Threshold Sweeping Analysis:
By lowering the decision threshold on **Logistic Regression (Balanced)**, we can increase the Test Recall significantly:
- At **t = 0.40**, Test Recall reaches **80.85%** (identifying 38 of 47 test attrition cases) with **Precision = 36.89%** and **F1-Score = 50.67%**. ROC-AUC remains at **80.07%**.
- At **t = 0.30**, Test Recall reaches **82.98%**, but Precision drops below 30% (29.10%).

### Recommendation:
Depending on HR requirements:
- If the priority is **maximum detection rate (Recall)**: We recommend **Logistic Regression (Balanced) with a threshold of 0.40**. It captures 80.85% of at-risk employees, which is critical for prevention, while maintaining a reasonable precision of 36.89%.
- If the priority is a **balanced, highly-efficient workflow (F1-score)**: We recommend **Random Forest (Regularized)**, which achieves 68.09% Test Recall, 45.07% Precision, and a 54.24% F1-score with very stable generalization.

## 9. Save Recommended Pipeline
We save the balanced Logistic Regression pipeline to `models/attrition_pipeline.joblib` (without overwriting models/v1/ yet).

In [9]:
# By default, save balanced Logistic Regression pipeline to root models directory
joblib.dump(lr_pipeline, os.path.join("models", "attrition_pipeline.joblib"))
print("Successfully saved attrition pipeline to models/attrition_pipeline.joblib")

Successfully saved attrition pipeline to models/attrition_pipeline.joblib
